# Synthetic E-commerce Data Generator for WooCommerce

Generates realistic products, customers, and orders and pushes them into a live WooCommerce store via the REST API, so the store behaves like it has real, live traffic. Meant to feed a downstream Spark Structured Streaming pipeline (poll the Orders API -> land as JSON -> stream into Spark).

**Setup**
1. In WooCommerce Admin: Settings -> Advanced -> REST API -> Add Key. Give it Read/Write permissions, copy the Consumer Key and Secret.
2. Run the pip install cell below.
3. Fill in your credentials in the config cell (or better, set them as environment variables before launching Jupyter, and leave the fallback blank).
4. Run cells top to bottom. The last cell starts the live order loop — since notebooks can't easily do Ctrl+C mid-cell, set `n_orders` to a finite number for a notebook run, or use the kernel's Interrupt button to stop an indefinite run.


In [6]:
!pip install Faker woocommerce requests -q


[notice] A new release of pip is available: 24.0 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [7]:
from dotenv import load_dotenv
load_dotenv()

True

In [8]:
import os
import random
import time
from datetime import datetime

from faker import Faker
from woocommerce import API

# en_IN gives Indian names, addresses, PIN codes and phone number formats.
# Change the locale here if your store targets a different market.
fake = Faker("en_IN")

# ---- Config: fill these in, or set as environment variables before launching Jupyter ----
WC_URL = os.environ.get("WC_URL", "")              # e.g. "https://yourstore.com"
WC_CONSUMER_KEY = os.environ.get("WC_CONSUMER_KEY", "")
WC_CONSUMER_SECRET = os.environ.get("WC_CONSUMER_SECRET", "")

if not all([WC_URL, WC_CONSUMER_KEY, WC_CONSUMER_SECRET]):
    raise EnvironmentError(
        "Set WC_URL, WC_CONSUMER_KEY, WC_CONSUMER_SECRET (env vars or directly above).\n"
        "Get keys from WooCommerce Admin -> Settings -> Advanced -> REST API."
    )

wcapi = API(
    url=WC_URL,
    consumer_key=WC_CONSUMER_KEY,
    consumer_secret=WC_CONSUMER_SECRET,
    version="wc/v3",
    timeout=20,
    verify_ssl=False,  # needed for LocalWP's self-signed local SSL certificate
)

import urllib3
urllib3.disable_warnings(urllib3.exceptions.InsecureRequestWarning)

In [9]:
print("Key:", repr(WC_CONSUMER_KEY))
print("Secret:", repr(WC_CONSUMER_SECRET))
print("URL:", repr(WC_URL))

Key: 'ck_854909d96800af1a046423e0a29675d148d3be7b'
Secret: 'cs_470f6473f81f7584243b9923b83df56744f97409'
URL: 'https://retail-analytics.local'


## Connectivity check
Run this before anything else. It confirms your URL, permalinks, and API key/secret are all correct, without creating any data yet.

In [10]:
resp = wcapi.get("products", params={"per_page": 1})
print("Status code:", resp.status_code)

if resp.status_code == 200:
    print("Auth OK. Sample response:", resp.json())
elif resp.status_code == 401:
    print("401 Unauthorized -> check Consumer Key/Secret, and that Permissions is Read/Write.")
elif resp.status_code == 404:
    print("404 Not Found -> check WC_URL is just the base site URL, and Permalinks isn't set to 'Plain'.")
else:
    print("Unexpected response:", resp.text[:300])

Status code: 200
Auth OK. Sample response: [{'id': 900, 'name': 'Illo Shampoo Bar', 'slug': 'illo-shampoo-bar', 'permalink': 'https://retail-analytics.local/product/illo-shampoo-bar/', 'date_created': '2026-07-28T07:22:33', 'date_created_gmt': '2026-07-28T07:22:33', 'date_modified': '2026-07-28T07:24:41', 'date_modified_gmt': '2026-07-28T07:24:41', 'type': 'simple', 'status': 'publish', 'featured': False, 'catalog_visibility': 'visible', 'description': '<p>Rem alias incidunt accusantium aperiam voluptate nobis.</p>\n', 'short_description': '<p>Pariatur tempora architecto tenetur sit ullam blanditiis.</p>\n', 'sku': '', 'price': '4819.27', 'regular_price': '4819.27', 'sale_price': '', 'date_on_sale_from': None, 'date_on_sale_from_gmt': None, 'date_on_sale_to': None, 'date_on_sale_to_gmt': None, 'on_sale': False, 'purchasable': True, 'total_sales': 6, 'virtual': False, 'downloadable': False, 'downloads': [], 'download_limit': -1, 'download_expiry': -1, 'external_url': '', 'button_text': 

## Domain data
Swap or expand these for a different retail niche.

In [11]:
CATEGORIES = {
    "Electronics": ["Wireless Earbuds", "Bluetooth Speaker", "Smartwatch", "Power Bank", "USB-C Cable", "Laptop Stand"],
    "Apparel": ["Cotton T-Shirt", "Denim Jacket", "Running Shoes", "Wool Sweater", "Baseball Cap", "Canvas Backpack"],
    "Home & Kitchen": ["Ceramic Mug Set", "Non-Stick Pan", "LED Desk Lamp", "Throw Pillow", "Cutting Board", "Air Fryer"],
    "Beauty": ["Face Serum", "Lip Balm Set", "Hair Dryer", "Makeup Brush Set", "Sunscreen SPF50", "Shampoo Bar"],
}

ORDER_STATUS_FLOW = ["pending", "processing", "completed"]

## Product creation

In [12]:
def create_products(n_per_category=5):
    """Create synthetic products across categories. Returns list of dicts with id/price."""
    created = []
    for category, names in CATEGORIES.items():
        for name in random.sample(names, min(n_per_category, len(names))):
            price = round(random.uniform(8, 250) * 80, 2)  # rough INR scaling
            payload = {
                "name": f"{fake.word().capitalize()} {name}",
                "type": "simple",
                "regular_price": str(price),
                "description": fake.paragraph(nb_sentences=3),
                "short_description": fake.sentence(),
                "categories": [{"name": category}],
                "manage_stock": True,
                "stock_quantity": random.randint(20, 200),
            }
            try:
                resp = wcapi.post("products", payload)
            except Exception as e:
                print(f"[product] request error: {e}")
                continue
            if resp.status_code == 201:
                product = resp.json()
                created.append({"id": product["id"], "price": price, "name": payload["name"]})
                print(f"[product] created #{product['id']} - {payload['name']} (Rs.{price})")
            else:
                print(f"[product] FAILED: {resp.status_code} {resp.text[:200]}")
    return created

## Customer creation

In [13]:
def create_customer():
    """Create one synthetic customer. Returns a dict with id and billing info, or None on failure."""
    first, last = fake.first_name(), fake.last_name()
    billing = {
        "first_name": first,
        "last_name": last,
        "address_1": fake.street_address(),
        "city": fake.city(),
        "state": fake.state(),
        "postcode": fake.postcode(),
        "country": "IN",
        "email": fake.email(),
        "phone": fake.phone_number(),
    }
    payload = {
        "email": fake.unique.email(),
        "first_name": first,
        "last_name": last,
        "username": f"{first.lower()}{random.randint(100, 999)}",
        "billing": billing,
    }
    try:
        resp = wcapi.post("customers", payload)
    except Exception as e:
        print(f"[customer] request error: {e}")
        return None
    if resp.status_code == 201:
        customer = resp.json()
        print(f"[customer] created #{customer['id']} - {first} {last}")
        return {"id": customer["id"], "billing": billing}
    print(f"[customer] FAILED: {resp.status_code} {resp.text[:200]}")
    return None

## Order creation and lifecycle

In [14]:
def create_order(customer, products):
    """Create one synthetic order for a customer, including their billing/shipping address."""
    n_items = random.randint(1, 4)
    chosen = random.sample(products, min(n_items, len(products)))
    line_items = [{"product_id": p["id"], "quantity": random.randint(1, 3)} for p in chosen]

    payload = {
        "customer_id": customer["id"],
        "billing": customer["billing"],
        "shipping": {k: v for k, v in customer["billing"].items() if k != "email"},
        "payment_method": random.choice(["cod", "bacs", "cheque"]),
        "payment_method_title": random.choice(["Cash on Delivery", "Direct Bank Transfer", "Cheque"]),
        "set_paid": random.choice([True, False]),
        "status": "pending",
        "line_items": line_items,
    }
    try:
        resp = wcapi.post("orders", payload)
    except Exception as e:
        print(f"[order] request error: {e}")
        return None
    if resp.status_code == 201:
        order = resp.json()
        print(f"[order] created #{order['id']} - customer {customer['id']}, {len(line_items)} items, total Rs.{order['total']}")
        return order["id"]
    print(f"[order] FAILED: {resp.status_code} {resp.text[:200]}")
    return None


def advance_order_status(order_id):
    """Progress an order to the next lifecycle status, simulating real fulfillment."""
    idx = random.randint(0, len(ORDER_STATUS_FLOW) - 2)
    next_status = ORDER_STATUS_FLOW[idx + 1]
    try:
        resp = wcapi.put(f"orders/{order_id}", {"status": next_status})
    except Exception as e:
        print(f"[order] status update error: {e}")
        return
    if resp.status_code == 200:
        print(f"[order] #{order_id} -> {next_status}")

## Main loop
Seeds products + customers, then generates orders on a timer.

**Notebook note:** set `n_orders` to a finite number (e.g. 30) for a normal run-to-completion in a cell. Leave it `None` only if you plan to stop it with the kernel's Interrupt/Stop button.

In [15]:
def run(n_customers=15, n_products_per_category=5, interval_range=(5, 20), n_orders=30):
    print(f"[{datetime.now()}] Seeding products and customers...")
    products = create_products(n_products_per_category)
    customers = [c for c in (create_customer() for _ in range(n_customers)) if c]

    if not products or not customers:
        print("No products/customers created - check API credentials and key permissions.")
        return

    print(f"[{datetime.now()}] Starting order stream...")
    count = 0
    open_orders = []
    try:
        while n_orders is None or count < n_orders:
            customer = random.choice(customers)
            order_id = create_order(customer, products)
            if order_id:
                open_orders.append(order_id)
                count += 1

            # Occasionally progress an older order's status to simulate fulfillment.
            if open_orders and random.random() < 0.4:
                advance_order_status(random.choice(open_orders))

            time.sleep(random.uniform(*interval_range))
    except KeyboardInterrupt:
        print(f"\nStopped. Created {count} orders this session.")

In [ ]:
run(n_orders=None)

[2026-07-28 13:05:02.754948] Seeding products and customers...
[product] created #911 - Maxime Bluetooth Speaker (Rs.15860.09)
[product] created #912 - Voluptas Power Bank (Rs.7733.6)
[product] created #913 - Eveniet Laptop Stand (Rs.7245.37)
[product] created #914 - Quia USB-C Cable (Rs.2742.76)
[product] created #915 - Eligendi Wireless Earbuds (Rs.10067.15)
[product] created #916 - Vel Baseball Cap (Rs.761.49)
[product] created #917 - Assumenda Cotton T-Shirt (Rs.17207.32)
[product] created #918 - Necessitatibus Wool Sweater (Rs.8301.85)
[product] created #919 - Temporibus Running Shoes (Rs.17651.98)
[product] created #920 - In Canvas Backpack (Rs.11738.78)
[product] created #921 - Cumque Non-Stick Pan (Rs.13003.04)
[product] created #922 - Temporibus Cutting Board (Rs.14624.22)
[product] created #923 - Culpa LED Desk Lamp (Rs.16642.79)
[product] created #924 - Quasi Ceramic Mug Set (Rs.9002.14)
[product] created #925 - Maiores Air Fryer (Rs.7565.49)
[product] created #926 - Sequi S

In [ ]:
import json

resp = wcapi.get("orders", params={"per_page": 1, "orderby": "date", "order": "desc"})
print(json.dumps(resp.json()[0], indent=2))

{
  "id": 111,
  "parent_id": 0,
  "status": "pending",
  "currency": "INR",
  "version": "10.9.4",
  "prices_include_tax": false,
  "date_created": "2026-07-26T14:49:37",
  "date_modified": "2026-07-26T14:49:37",
  "discount_total": "0.00",
  "discount_tax": "0.00",
  "shipping_total": "0.00",
  "shipping_tax": "0.00",
  "cart_tax": "0.00",
  "total": "27886.71",
  "total_tax": "0.00",
  "customer_id": 31,
  "order_key": "wc_order_f0GvqMs7noZqg",
  "billing": {
    "first_name": "Sneha",
    "last_name": "Kaur",
    "company": "",
    "address_1": "48/728\nSastry Chowk",
    "address_2": "",
    "city": "Vasai-Virar",
    "state": "Haryana",
    "postcode": "959412",
    "country": "IN",
    "email": "chakrika14@example.net",
    "phone": "+911144101731"
  },
  "shipping": {
    "first_name": "Sneha",
    "last_name": "Kaur",
    "company": "",
    "address_1": "48/728\nSastry Chowk",
    "address_2": "",
    "city": "Vasai-Virar",
    "state": "Haryana",
    "postcode": "959412",
   